# **Data Cleaning and Preparation**

## **Introduction:**

In this project, we will take the next step and prepare the data for machine learning.
We will clean the data where necessary, create useful features from the existing stock information, combine the datasets, and prepare a final dataset that can be used in the next project.

## **Load the dataset** 

In [2]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Load the datasets

stock_data = pd.read_csv(
    "../Project_1_EDA_Stock_Market_Data/historical_stocks.csv")

stock_price_data = pd.read_csv(
    "../Project_1_EDA_Stock_Market_Data/historical_stock_prices.csv")

In [4]:
print(stock_data.shape)
print(stock_price_data.shape)

(6460, 5)
(20973889, 8)


In [5]:
stock_data.head()

,ticker,exchange,name,sector,industry
0,PIH,NASDAQ,"1347 PROPERTY INSURANCE HOLDINGS, INC.",FINANCE,PROPERTY-CASUALTY INSURERS
1,PIHPP,NASDAQ,"1347 PROPERTY INSURANCE HOLDINGS, INC.",FINANCE,PROPERTY-CASUALTY INSURERS
2,TURN,NASDAQ,180 DEGREE CAPITAL CORP.,FINANCE,FINANCE/INVESTORS SERVICES
3,FLWS,NASDAQ,"1-800 FLOWERS.COM, INC.",CONSUMER SERVICES,OTHER SPECIALTY STORES
4,FCCY,NASDAQ,1ST CONSTITUTION BANCORP (NJ),FINANCE,SAVINGS INSTITUTIONS


In [6]:
stock_price_data.head()

,ticker,open,close,adj_close,low,high,volume,date
0,AHH,11.50,11.58,8.493155,11.25,11.68,4633900,2013-05-08
1,AHH,11.66,11.55,8.471151,11.50,11.66,275800,2013-05-09
2,AHH,11.55,11.60,8.507822,11.50,11.60,277100,2013-05-10
3,AHH,11.63,11.65,8.544494,11.55,11.65,147400,2013-05-13
4,AHH,11.60,11.53,8.456484,11.50,11.60,184100,2013-05-14


In [13]:
stock_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 6460 entries, 0 to 6459
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   ticker    6460 non-null   str  
 1   exchange  6460 non-null   str  
 2   name      6460 non-null   str  
 3   sector    5020 non-null   str  
 4   industry  5020 non-null   str  
dtypes: str(5)
memory usage: 640.9 KB


In [12]:
stock_price_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 20973889 entries, 0 to 20973888
Data columns (total 8 columns):
 #   Column     Dtype  
---  ------     -----  
 0   ticker     str    
 1   open       float64
 2   close      float64
 3   adj_close  float64
 4   low        float64
 5   high       float64
 6   volume     int64  
 7   date       str    
dtypes: float64(5), int64(1), str(2)
memory usage: 1.5 GB


## **Part 1 — Clean ad prepare the Data**

### **Clean the data**

In [17]:
# Convert date to datetime format

stock_price_data["date"] = pd.to_datetime(stock_price_data["date"])

In [18]:
print(stock_price_data["date"].dtype)

datetime64[us]


In [ ]:
# Remove the header row inserted in the data

stock_data = stock_data[stock_data["ticker"] != "Symbol"].copy()

In [20]:
# Sort the stock-price data by ticker and date 

stock_price_data = stock_price_data.sort_values(by=["ticker", "date"], ascending=True)

stock_price_data.head()

,ticker,open,close,adj_close,low,high,volume,date
13766141,A,32.546494,31.473534,27.494957,28.612303,35.765381,62546300,1999-11-18
13766149,A,30.713520,28.880543,25.229753,28.478184,30.758226,15234100,1999-11-19
13766150,A,29.551144,31.473534,27.494957,28.657009,31.473534,6577800,1999-11-22
13766151,A,30.400572,28.612303,24.995413,28.612303,31.205294,5975600,1999-11-23
13766152,A,28.701717,29.372318,25.659359,28.612303,29.998211,4843200,1999-11-24


### **Handle Unusual Values**

In [ ]:
# Review unusual close values

stock_price_data.nlargest(15, "close")[
    ["ticker", "date", "close", "adj_close", "volume"]]

,ticker,date,close,adj_close,volume
12146443,SCON,2000-02-28,1779750.0,1779750.0,400
12146459,SCON,2000-02-29,1595250.0,1595250.0,400
12146546,SCON,2000-03-13,1363500.0,1363500.0,300
17889354,CODA,2001-02-05,1354500.0,1354500.0,34
2444619,TVIX,2013-02-26,1347500.0,1347500.0,100
12146480,SCON,2000-03-02,1261125.0,1261125.0,200
12146437,SCON,2000-02-25,1215000.0,1215000.0,200
17107201,TOPS,2016-08-01,1189800.0,1189800.0,100
12146465,SCON,2000-03-01,1134000.0,1134000.0,300
12146562,SCON,2000-03-14,1111500.0,1111500.0,100


In [ ]:
# Number of anomalies

aan_anomaly = (
    (stock_price_data["ticker"] == "AAN") &
    (stock_price_data["adj_close"] > 1_000_000))

aan_anomaly.sum()

np.int64(824)

In [ ]:
# Replace anomalies with NaN

stock_price_data.loc[aan_anomaly, "adj_close"] = np.nan

### **Create basic features for price data**

In [ ]:
# Remove abnormal header row

stock_data

### **Insights**:

We also reviewed the unusually high close values. Although some closing prices were extremely high, the corresponding adj_close values were similar, suggesting that these observations were not clearly data errors. Therefore, we retained because there was insufficient evidence to consider them data errors.

We reviewed the unusual adj_close values identified in Project 1 and confirmed that 824 AAN observations had adjusted closing prices above $1,000,000. These values were considered anomalous because they were extremely different from the corresponding closing prices.We replaced only these anomalous adj_close values with NaN rather than deleting the entire rows. This allowed us to retain the other useful information in those records while avoiding the use of unreliable adjusted closing prices.

## **Part 2 — Create Useful Features**

In [27]:
# Create price change feature

stock_price_data["price_change"] = (
    stock_price_data.groupby("ticker")["close"].diff())

In [ ]:
# Create the daily return feature

stock_price_data["daily_return"] = (stock_price_data.groupby("ticker")["close"].pct_change())

In [31]:
# Create a close_lag

stock_price_data["close_lag_1"] = (stock_price_data.groupby("ticker")["close"].shift(1))

In [32]:
stock_price_data.head()

,ticker,open,close,adj_close,low,high,volume,date,price_change,daily_return,close_lag_1
13766141,A,32.546494,31.473534,27.494957,28.612303,35.765381,62546300,1999-11-18,NaN,NaN,NaN
13766149,A,30.713520,28.880543,25.229753,28.478184,30.758226,15234100,1999-11-19,-2.592991,-0.082386,31.473534
13766150,A,29.551144,31.473534,27.494957,28.657009,31.473534,6577800,1999-11-22,2.592991,0.089783,28.880543
13766151,A,30.400572,28.612303,24.995413,28.612303,31.205294,5975600,1999-11-23,-2.861231,-0.090909,31.473534
13766152,A,28.701717,29.372318,25.659359,28.612303,29.998211,4843200,1999-11-24,0.760015,0.026563,28.612303


In [ ]:
# Compute the moving average with window 5

stock_price_data["ma_5"] = (stock_price_data.groupby("ticker")["close"]
                          .transform(lambda x:x.rolling(5).mean()))

In [34]:
# Compute the moving average with window 20

stock_price_data["ma_20"] = (stock_price_data.groupby("ticker")["close"]
                          .transform(lambda x: x.rolling(20).mean()))

In [35]:
stock_price_data.head()

,ticker,open,close,adj_close,low,high,volume,date,price_change,daily_return,close_lag_1,ma_5,ma_20
13766141,A,32.546494,31.473534,27.494957,28.612303,35.765381,62546300,1999-11-18,NaN,NaN,NaN,NaN,NaN
13766149,A,30.713520,28.880543,25.229753,28.478184,30.758226,15234100,1999-11-19,-2.592991,-0.082386,31.473534,NaN,NaN
13766150,A,29.551144,31.473534,27.494957,28.657009,31.473534,6577800,1999-11-22,2.592991,0.089783,28.880543,NaN,NaN
13766151,A,30.400572,28.612303,24.995413,28.612303,31.205294,5975600,1999-11-23,-2.861231,-0.090909,31.473534,NaN,NaN
13766152,A,28.701717,29.372318,25.659359,28.612303,29.998211,4843200,1999-11-24,0.760015,0.026563,28.612303,29.962446,NaN
